# Classificazione nei Layer Lamfalussy

Questo notebook assegna a ciascun nodo del grafo focale un **layer funzionale** secondo il framework Lamfalussy adattato:

| Layer | Definizione | Esempi |
|---|---|---|
| **L1** | Legislazione quadro — principi e obiettivi generali | Reg. PE e Consiglio, Direttive PE e Consiglio, Trattati |
| **L2** | Misure tecniche di attuazione — dettagli e procedure | Reg. delegati, Reg. di esecuzione, Decisioni della Commissione |
| **L3** | Soft law — linee guida, raccomandazioni, interpretazioni | Guidelines EBA/ESMA, Raccomandazioni, Comunicazioni |
| **L4** | Enforcement — controllo, giurisprudenza, infrazioni | Sentenze CGUE, Decisioni di infrazione |

## Metodologia di classificazione

La classificazione avviene in **due fasi**:

1. **Regole deterministiche sul titolo** (per i nodi con titolo): pattern testuali derivati dalla struttura formale degli atti UE. Ad esempio, la presenza di `DELEGATED` nel titolo identifica univocamente un atto L2 ai sensi dell'art. 290 TFUE.

2. **Fallback sui metadati** (per i nodi senza titolo): classificazione basata su `LegalType` e struttura CELEX, con certezza ridotta documentata nella colonna `layer_confidence`.

## Input/Output
- **Input**: `data/output/golden_power/gephi_nodes_focal_titled.csv`
- **Output**: `data/output/golden_power/gephi_nodes_focal_layered.csv`

## 0. Setup

In [ ]:
import pandas as pd
import os
import sys
import re

sys.path.append('..')
from config_golden_power import MATERIA_NAME

output_path = os.path.join('..', 'data', 'output', MATERIA_NAME)
input_file  = os.path.join(output_path, 'gephi_nodes_focal_titled.csv')
output_file = os.path.join(output_path, 'gephi_nodes_focal_layered.csv')

nodes = pd.read_csv(input_file)

print(f"Nodi caricati: {len(nodes)}")
print(f"Con titolo:    {nodes['title'].notna().sum()} ({nodes['title'].notna().sum()/len(nodes)*100:.1f}%)")
print(f"Senza titolo:  {nodes['title'].isna().sum()} ({nodes['title'].isna().sum()/len(nodes)*100:.1f}%)")

## 1. Regole Deterministiche sul Titolo

Le regole sono ordinate per **priorità decrescente**: una regola più specifica ha precedenza su una più generica.

### Fondamento normativo delle regole

- **L2 Delegated**: TFUE art. 290 — "atti delegati" adottati dalla Commissione su delega del legislatore
- **L2 Implementing**: TFUE art. 291 — "atti di esecuzione" adottati dalla Commissione
- **L1 Parliament+Council**: atti adottati con procedura legislativa ordinaria (TFUE art. 294)
- **L1 Council**: atti del Consiglio in materie di competenza esclusiva o procedura speciale
- **L2 Commission alone**: regolamenti/decisioni della sola Commissione = atti non legislativi
- **L3 Guidelines/Recommendation**: soft law, non vincolante per definizione
- **L4 Judgment/Court**: giurisprudenza CGUE

In [ ]:
def classify_by_title(title):
    """
    Classifica un atto nel framework Lamfalussy in base al titolo.

    Restituisce (layer, confidence, rule) dove:
    - layer:      'L1', 'L2', 'L3', 'L4'
    - confidence: 'high', 'medium'
    - rule:       stringa che identifica la regola applicata (per audit)
    """
    if pd.isna(title) or title == '':
        return None, None, 'no_title'

    t = title.upper()

    # ----------------------------------------------------------------
    # L4 — Giurisprudenza (massima priorità: non ambigua)
    # ----------------------------------------------------------------
    if any(x in t for x in ['JUDGMENT OF THE COURT', 'ORDER OF THE COURT',
                             'OPINION OF THE COURT', 'RULING OF THE COURT']):
        return 'L4', 'high', 'court_judgment'

    # ----------------------------------------------------------------
    # L2 — Atti delegati e di esecuzione (TFUE artt. 290-291)
    # Priorità alta: la parola DELEGATED/IMPLEMENTING è inequivocabile
    # ----------------------------------------------------------------
    if 'DELEGATED' in t:
        return 'L2', 'high', 'delegated_act'

    if 'IMPLEMENTING' in t:
        return 'L2', 'high', 'implementing_act'

    # ----------------------------------------------------------------
    # L1 — Atti legislativi del PE e del Consiglio
    # ----------------------------------------------------------------
    if 'EUROPEAN PARLIAMENT AND OF THE COUNCIL' in t:
        return 'L1', 'high', 'parliament_and_council'

    if re.search(r'COUNCIL (REGULATION|DIRECTIVE|DECISION|FRAMEWORK DECISION)', t):
        return 'L1', 'high', 'council_act'

    # ----------------------------------------------------------------
    # L3 — Soft law
    # ----------------------------------------------------------------
    if any(x in t for x in ['GUIDELINE', 'GUIDELINES']):
        return 'L3', 'high', 'guidelines'

    if 'RECOMMENDATION' in t:
        return 'L3', 'high', 'recommendation'

    if 'COMMUNICATION FROM THE COMMISSION' in t:
        return 'L3', 'medium', 'commission_communication'

    if any(x in t for x in ['OPINION OF THE', 'OPINION OF THE EUROPEAN']):
        return 'L3', 'medium', 'opinion'

    # ----------------------------------------------------------------
    # L2 — Atti della sola Commissione (non legislativi per definizione)
    # ----------------------------------------------------------------
    if re.search(r'COMMISSION (REGULATION|DECISION|DIRECTIVE)', t):
        return 'L2', 'medium', 'commission_act'

    # ----------------------------------------------------------------
    # L1 — Trattati (identificati dal titolo)
    # ----------------------------------------------------------------
    if any(x in t for x in ['TREATY ON', 'TREATY ESTABLISHING',
                             'TREATY OF', 'CONSOLIDATED VERSION OF THE TREATY']):
        return 'L1', 'high', 'treaty_title'

    return None, None, 'unmatched'


# Applica le regole
result = nodes['title'].apply(classify_by_title)
nodes['layer_title']      = result.apply(lambda x: x[0])
nodes['layer_confidence'] = result.apply(lambda x: x[1])
nodes['layer_rule']       = result.apply(lambda x: x[2])

print("Risultati classificazione da titolo:")
print(nodes['layer_title'].value_counts(dropna=False).to_string())
print()
print("Regole applicate:")
print(nodes['layer_rule'].value_counts().to_string())

## 2. Fallback sui Metadati

Per i nodi senza titolo o con titolo non classificabile (`unmatched`, `no_title`), usiamo `LegalType` e la struttura del CELEX come proxy.

| LegalType | Layer fallback | Certezza | Motivazione |
|---|---|---|---|
| `Treaty` | L1 | Alta | Diritto primario per definizione |
| `Case_Law` | L4 | Alta | Giurisprudenza per definizione |
| `Legislative_Act` | L1 | Media | Atti legislativi non altrimenti classificati |
| `Regulation` / `Directive` | L1_or_L2 | Bassa | Ambiguo senza titolo |
| `Decision` | L2 | Bassa | La maggioranza delle decisioni sono esecutive |

In [ ]:
def classify_by_metadata(row):
    """
    Fallback: classifica in base a LegalType e CELEX
    quando il titolo non è disponibile o non è classificabile.
    """
    legal_type = row['LegalType']
    celex      = str(row['Label']) if pd.notna(row['Label']) else ''

    # Trattati: sempre L1
    if legal_type == 'Treaty':
        return 'L1', 'high', 'fallback_treaty'

    # Giurisprudenza: sempre L4
    if legal_type == 'Case_Law':
        return 'L4', 'high', 'fallback_caselaw'

    # Legislative_Act: probabile L1
    if legal_type == 'Legislative_Act':
        return 'L1', 'medium', 'fallback_legislative_act'

    # Regulation/Directive/Decision: ambiguo
    # Proxy parziale: settore CELEX 3 + anno recente + tipo R
    # I regolamenti delegati post-2010 hanno spesso numeri bassi
    # Ma non è affidabile — marchiamo come low confidence
    if legal_type in ('Regulation', 'Directive'):
        return 'L1', 'low', 'fallback_reg_dir'

    if legal_type == 'Decision':
        return 'L2', 'low', 'fallback_decision'

    return 'L1', 'low', 'fallback_unknown'


# Applica fallback solo dove il titolo non ha dato risultati
needs_fallback = nodes['layer_title'].isna()

fallback_result = nodes[needs_fallback].apply(classify_by_metadata, axis=1)
nodes.loc[needs_fallback, 'layer_title']      = fallback_result.apply(lambda x: x[0])
nodes.loc[needs_fallback, 'layer_confidence'] = fallback_result.apply(lambda x: x[1])
nodes.loc[needs_fallback, 'layer_rule']       = fallback_result.apply(lambda x: x[2])

print(f"Nodi che hanno usato il fallback: {needs_fallback.sum()}")
print()
print("Regole di fallback applicate:")
print(nodes[needs_fallback]['layer_rule'].value_counts().to_string())

## 3. Colonna Layer Finale

Consolidiamo in un'unica colonna `Layer` e aggiungiamo una colonna `layer_source` che indica se la classificazione viene dal titolo o dal fallback — utile per la sezione metodologica della tesi e per analisi di sensibilità.

In [ ]:
nodes['Layer'] = nodes['layer_title']

nodes['layer_source'] = nodes['layer_rule'].apply(
    lambda r: 'title'    if pd.notna(r) and not r.startswith('fallback') and r != 'no_title' and r != 'unmatched'
         else 'fallback' if pd.notna(r) and r.startswith('fallback')
         else 'unresolved'
)

print("=== DISTRIBUZIONE LAYER FINALE ===")
print()
print("Per layer:")
print(nodes['Layer'].value_counts().to_string())
print()
print("Per fonte di classificazione:")
print(nodes['layer_source'].value_counts().to_string())
print()
print("Per layer e confidence:")
print(pd.crosstab(nodes['Layer'], nodes['layer_confidence']).to_string())

## 4. Verifica Qualitativa

Campione di atti per ciascun layer — verifica visiva che la classificazione sia sensata.

In [ ]:
for layer in ['L1', 'L2', 'L3', 'L4']:
    subset = nodes[
        (nodes['Layer'] == layer) &
        (nodes['title'].notna()) &
        (nodes['layer_source'] == 'title')
    ]['title'].head(3)

    print(f"--- {layer} ---")
    for t in subset:
        print(f"  {t[:100]}")
    print()

## 5. Export

In [ ]:
nodes.to_csv(output_file, index=False)

print(f"File salvato: {output_file}")
print(f"Colonne: {nodes.columns.tolist()}")
print()
print("=== RIEPILOGO METODOLOGICO ===")
print(f"Nodi totali:                    {len(nodes)}")
print(f"Classificati da titolo:         {(nodes['layer_source'] == 'title').sum()} ({(nodes['layer_source'] == 'title').sum()/len(nodes)*100:.1f}%)")
print(f"Classificati da fallback:       {(nodes['layer_source'] == 'fallback').sum()} ({(nodes['layer_source'] == 'fallback').sum()/len(nodes)*100:.1f}%)")
print(f"Non risolti:                    {(nodes['layer_source'] == 'unresolved').sum()}")
print()
print("Confidence per layer:")
print(pd.crosstab(nodes['Layer'], nodes['layer_confidence'],
                  margins=True, margins_name='Totale').to_string())